[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C38_Frameworks_Accel_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy **从零复刻框架内核**（autograd / 编译器 / 函数变换 / 混合精度 / profiler），真实 **PyTorch/JAX** 代码作 **对照**。

这个 notebook 做四件事：① 确认环境（含**优雅检测 torch/jax 是否可用**，缺了不阻断）；② 用一个最小例子体会「**框架替你做了什么**」；③ 立下全课的纪律——**对拍（differential testing）**，特别是**数值梯度**这个 autograd 的黄金参考；④ 预览**真实 API 对照**的读法。

> 心智模型：**框架 = autograd + 编译 + 变换 + 精度/显存管理 + 观测**。本课把每一块的*内核*用 numpy 剥出来重写。

## 1 · 环境自检（torch/jax 可选，优雅回退）

只需要 `numpy`。`torch`/`jax` **可选**——有则可跑少数「真实 API 对照」cell，无则当讲解读，**对任何练习与 assert 零影响**。
下面演示本课统一的**优雅检测**写法（`try/except`），它会贯穿所有对照 cell。

In [ ]:
import sys, platform
import numpy as np
print('Python', sys.version.split()[0], '|', platform.system())
print('numpy ', np.__version__, '(必需)')

# ——— 本课统一的可选依赖检测：缺了绝不阻断 ———
def try_import(name):
    try:
        mod = __import__(name)
        return mod, getattr(mod, '__version__', '?')
    except Exception:
        return None, None

torch, tver = try_import('torch')
jax,   jver = try_import('jax')
HAS_TORCH = torch is not None
HAS_JAX   = jax   is not None
print('torch ', tver if HAS_TORCH else '未安装 -> 对照cell将跳过(讲解形态)', '(可选)')
print('jax   ', jver if HAS_JAX   else '未安装 -> 对照cell将跳过(讲解形态)', '(可选)')

assert np is not None, '本课只硬性依赖 numpy'
print('\n环境就绪 ✅  （core 路径纯 numpy，torch/jax 仅用于可选对照）')

## 2 · 框架替你做了什么？一个最小对照

用 numpy 手写一个线性层 `y = x @ W + b` 的前向**和**反向（你已经会了），再看「框架视角」：你只写前向，`backward()` 自动给出所有梯度。本节先用 numpy 把这件事做出来，建立「autograd 要复刻的目标」。

In [ ]:
rng = np.random.default_rng(0)
x = rng.standard_normal((4, 3))      # batch=4, in=3
W = rng.standard_normal((3, 2))      # in=3, out=2
b = rng.standard_normal((2,))

# 前向 + 一个标量 loss = sum(y)
y = x @ W + b
loss = y.sum()

# 手推反向（链式法则）：dloss/dy = 1
dy = np.ones_like(y)                  # (4,2)
dW = x.T @ dy                         # (3,2)
db = dy.sum(axis=0)                   # (2,)  <- 广播梯度：沿 batch 维 sum 回去！
dx = dy @ W.T                         # (4,3)

print('loss =', round(float(loss), 4))
print('dW shape', dW.shape, '| db shape', db.shape, '| dx shape', dx.shape)
# 关键观察：b 是 (2,) 被广播到 (4,2)，其梯度必须沿被广播的 batch 维求和
assert db.shape == b.shape and dW.shape == W.shape and dx.shape == x.shape
print('✅ 手写前向+反向通过。模块 01 的 autograd 要做的，就是让这套反向【自动】发生。')

**关键观察**：偏置 `b` 形状 `(2,)` 被广播到 `(4,2)`，它的梯度必须**沿被广播的 batch 维求和**回 `(2,)`。

这个「广播 → 反向 sum 回去」是手写 autograd 最容易错的地方，也是模块 01 的核心细节之一。框架替你自动处理它。

## 3 · 立纪律之一：对拍数值梯度（autograd 的黄金参考）

怎么知道一个 autograd 实现对不对？用**数值梯度**（中心差分）：`g ≈ (f(x+ε) - f(x-ε)) / 2ε`。
它慢、有误差，但**与任何 autograd 实现无关**，是验证解析梯度的黄金参考。本课每个 autograd 都与它对拍。

In [ ]:
def numerical_grad(f, x, eps=1e-6):
    '''对标量函数 f: R^n -> R 在 x 处求数值梯度（中心差分）。'''
    x = np.asarray(x, dtype=float)
    g = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f(x)
        x[i] = old - eps; fm = f(x)
        x[i] = old
        g[i] = (fp - fm) / (2 * eps)
        it.iternext()
    return g

# 用 f(x) = sum( (x @ W) ** 2 ) 测试：解析梯度 = 2 (x@W) @ W.T
W2 = rng.standard_normal((3, 2))
def f(x):
    return float(((x @ W2) ** 2).sum())
x0 = rng.standard_normal((4, 3))
g_analytic = 2 * (x0 @ W2) @ W2.T
g_numeric  = numerical_grad(f, x0)
rel_err = np.abs(g_analytic - g_numeric).max() / (np.abs(g_analytic).max() + 1e-12)
print(f'解析 vs 数值梯度 相对误差 = {rel_err:.2e}')
assert rel_err < 1e-5, '解析梯度应与数值梯度一致'
print('✅ 数值梯度对拍通过 —— 这是模块 01 验证 autograd 正确性的统一裁判。')

## 4 · 立纪律之二：通用对拍工具

把「对拍」封装成一个小函数，后面每个模块都用它判定「我的内核 == 朴素参考」（融合前后结果一致、vmap == for 循环、等等）。

In [ ]:
def check_allclose(name, got, ref, atol=1e-10, rtol=1e-7):
    '''对拍：被测内核结果 vs 朴素参考。打印并 assert。'''
    got_a, ref_a = np.asarray(got, dtype=float), np.asarray(ref, dtype=float)
    ok = np.allclose(got_a, ref_a, atol=atol, rtol=rtol)
    max_err = float(np.max(np.abs(got_a - ref_a))) if got_a.size else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参考不一致！'
    return ok

# 演示：一个「分两步算」的内核 对拍 一次算完
A = rng.standard_normal((5, 5))
B = rng.standard_normal((5, 5))
fused   = A @ B + A          # 假想「融合」后
stepwise = (A @ B) + (A * 1) # 假想「融合」前（语义相同）
check_allclose('fuse vs stepwise', fused, stepwise)
print('\n这就是全课的工作流：写内核 -> 对拍参考 -> assert 兜底。')

## 5 · IEEE 754 速览：为什么会有 fp16/bf16 的取舍

模块 04 要用浮点位宽。先建立直觉：一个浮点数把位分给**指数**（决定动态范围）和**尾数**（决定相对精度）。
不同格式分法不同，造成 fp32/fp16/bf16 的取舍。下面用 numpy 把三种格式的**机器精度**实测出来。

In [ ]:
# 三种格式：(指数位, 尾数位)
FORMATS = {
    'fp32': (8, 23),
    'fp16': (5, 10),
    'bf16': (8, 7),
}
print(f"{'格式':>6} {'指数位':>6} {'尾数位':>6} {'机器eps(理论)':>16} {'最大指数范围':>14}")
for name, (ebits, mbits) in FORMATS.items():
    eps = 2.0 ** (-mbits)            # 机器精度 ~ 2^-尾数位
    max_exp = 2 ** (ebits - 1) - 1   # 指数范围 ~ 2^(指数位-1)
    print(f'{name:>6} {ebits:>6} {mbits:>6} {eps:>16.2e} {max_exp:>14}')
# numpy 直接验证 fp16/fp32 的机器 eps（cast 成 python float，避免 float16 比较下溢）
assert abs(float(np.finfo(np.float16).eps) - 2.0**-10) < 1e-9
assert abs(float(np.finfo(np.float32).eps) - 2.0**-23) < 1e-12
print('\n解读：fp16 指数位少(5) -> 范围窄、易溢出 -> 需 loss scaling；')
print('     bf16 指数位与 fp32 同(8) -> 范围大、通常不需 loss scaling，但尾数少(7) -> 精度低。')

**关键结论**：**指数位决定动态范围，尾数位决定相对精度**。

- `fp16`(5指数/10尾数)：精度尚可但**范围窄**（最大 ~65504），梯度易下溢 → 模块 04 的 **loss scaling** 就是来救它的。
- `bf16`(8指数/7尾数)：范围与 fp32 几乎相同（**通常无需 loss scaling**），但**精度低**。现代大模型训练的主流。

## 6 · 预览：真实 API 对照怎么读

本课每个 notebook 都会给**真实 PyTorch/JAX 对照**。读法：**先看懂上面的 numpy 内核，再把对照当成「同一件事在真框架里怎么写」**。
若装了 torch，下面这段会实跑并和我们的手写反向对拍；没装则打印讲解，**不报错**。

In [ ]:
# 真实 torch autograd 对照（装了 torch 才跑；没装则讲解，不阻断）
if HAS_TORCH:
    import torch
    xt = torch.tensor(x0, requires_grad=True)
    Wt = torch.tensor(W2)
    loss_t = ((xt @ Wt) ** 2).sum()
    loss_t.backward()                       # 自动反向！
    g_torch = xt.grad.numpy()
    err = np.abs(g_torch - g_analytic).max()
    print(f'torch.autograd vs 手写解析梯度 最大误差 = {err:.2e}')
    assert err < 1e-5
    print('✅ 真实 torch.autograd 与我们手写的反向一致 —— 模块 01 会复刻这个 .backward()')
else:
    print('（未装 torch）对照讲解：')
    print('  在 PyTorch 里，上面手写的反向就是：')
    print('    x = torch.tensor(x0, requires_grad=True)')
    print('    loss = ((x @ W) ** 2).sum()')
    print('    loss.backward()   # 自动算出 x.grad')
    print('  模块 01 将从零复刻这个 autograd 引擎。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 里复刻的每个框架内核（autograd / 融合 / vmap / loss-scaling / profiler），都会用 `np.allclose` 或数值梯度对拍参考；结构正确则数值一致，数值一致则逻辑可接回 torch/jax。

**接下来六个模块**：01 autograd → 02 torch.compile → 03 jax/xla → 04 混合精度与显存 → 05 profiling/调试。每一步都建立在前一步之上。

下一站：**模块 01 · 自动微分内核** —— 把 `loss.backward()` 从零写出来。